# SH classification-only full-market regression
Read-only audit of six actual full runs. Raw Parquet references, no window overrides. Pending runs are not passed. Run using Python with pyarrow from repository root or analysis/. Outputs are intentionally empty; summary.json is the live/final artifact.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'analysis').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / 'analysis'))
from run_sh_phase_revalidation import collect, duration
result = collect()
for r in result['rows']:
    t = r.get('timings', {}).get('stages', {})
    print(r['date'], r['status'], r.get('matched'), r.get('mismatched'), r.get('excluded_by_status'), duration(t.get('restore_total_seconds')), duration(t.get('validation_total_seconds')))


In [ ]:
if result['all_jobs_finished']:
    assert result['all_snapshot_validations_passed'], 'Inspect failed reports'
    assert result['classification_regression_passed'], 'Inspect classification/counter changes'
    changes = [x for r in result['rows'] for x in r['previous_anomalies']]
    assert len(changes) == 34
    assert all(x['current'] == 'excluded_by_status' for x in changes)
    print('All six full runs and classification regression passed.')
else:
    print('Still running: no final acceptance conclusion.')
